## 🌐 기획 배경 — 연결에서 시작해, 거래로 이어지는 글로벌 비즈니스 플랫폼

### 1. 해외 진출 시도는 계속 늘고 있다
2024년 기준 중소기업 수출은 **1,186억 달러**, 수출 참여 기업도 **약 9만 8천 개**로 역대 최대 수준이다.
본 분석에서도 2015 → 2023년 사이 **수출 참여 중소기업 +6.4%**, **교역액 +15.2%** 의 우상향 추세를 확인할 수 있다 *(Chart B)*.

> 출처: [Korea SME Exports Hit Record $118.6B in 2024 — Seoul Economic Daily, 2026.01](https://en.sedaily.com/finance/2026/01/28/korea-sme-exports-hit-record-1186-billion-in-2024)

### 2. 그러나 거래는 여전히 대기업에 쏠려 있다
활동기업의 **99.87%가 중소기업**이지만, 수출 교역액의 약 **65%는 대기업**이 차지한다 *(Chart A + C)*.
실제로 전체 수출의 약 39%가 상위 10대 기업에 집중되어 있어, 중소기업의 글로벌 판로 확보는 여전히 어려운 과제다.

> 출처: [South Korea's Export Top-10 Share Hits 39% — KoreaNewsfeed, 2026](https://www.reddit.com/r/KoreaNewsfeed/comments/1r0pedh/south_koreas_export_top_10_share_hits_39/)

### 3. 온라인이 답일 것 같지만, 실상은 다르다
온라인 수출은 증가 추세지만, **실제 활용 기업 비중은 약 4%** 수준에 머물러 있다.

> 출처: [Online Export Usage Stalls at 4% — 아시아경제, 2026.04](https://www.asiae.co.kr/en/print.htm?idxno=2026042910261400571)

### 4. 기존 B2B 거래는 비효율적이다
거래 한 건이 성사되기까지 **평균 200일 이상**이 걸리고, 절차도 복잡하다.

> 출처: [B2B eCommerce Statistics 2026 — elogic commerce](https://www.reddit.com/r/u_elogic__commerce/comments/1sq48wp/b2b_ecommerce_statistics_2026_market_size_buyer/)

### 5. 언어·물류·정보 비대칭이 부담을 더한다
언어 장벽, 물류 비용, 정보 격차가 겹치면서 기업 입장에서는 거래 자체가 큰 부담이 된다.

> 출처: [한국학술지인용색인 (KCI), ART003080628](https://www.kci.go.kr/kciportal/ci/sereArticleSearch/ciSereArtiView.kci?sereArticleSearchBean.artiId=ART003080628)

### 6. 시장 구조 자체가 분리되어 있다
거래는 플랫폼에서, 관계는 SNS에서 — **관계와 거래가 따로 움직이는 구조**다.
결국 기업은 매번 *"이 상대를 믿어도 되는지"* 를 스스로 검증해야 한다.

---

### 🎯 그래서 필요하다 — 관계가 거래로 이어지는 구조

처음부터 관계를 만들고, 그 신뢰가 자연스럽게 거래로 이어지는 구조.
이 프로젝트는 그 흐름을 만들기 위한 **피드 기반 비즈니스 소셜 마켓 플랫폼**에서 출발한다.

In [ ]:
# pip install openpyxl

### 공통 셋업

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform

# 한글 폰트 (Windows / Mac)
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid", font=plt.rcParams['font.family'])

### 데이터 로드

In [ ]:
# 데이터로드
df1 = pd.read_excel("./datasets/산업별_기업_규모별_기업수_활동_신생_소멸__20260504153638.xlsx", sheet_name="데이터_long_활동")
df2 = pd.read_excel("./datasets/기업규모별_수출강도별_수출입_20260504152906.xlsx", sheet_name="tidy")

In [ ]:
display(df1)
df1.shape

In [ ]:
df1.info()

## 1. 거래의 격차
기업수의 99.87%가 중소기업, 그러나 교역액의 약 65%는 대기업이 차지한다.

### 1-1. 기업수 비중 (2024 · 산업 전체)

In [ ]:
# 2024년 산업 전체에서 대기업/중소기업 활동기업 데이터 추출
count_slice = (
    df1[(df1['산업'] == '전체')
        & (df1['연도'] == 2024)
        & (df1['기업규모'].isin(['대기업', '중소기업']))]
    [['기업규모', '활동기업수']]
    .reset_index(drop=True)
)

display(count_slice)
count_slice.shape

In [ ]:
count_slice.info()

In [ ]:
# 결측치 확인
count_slice.isna().sum()

In [ ]:
# 중복행 확인
count_slice.duplicated().sum()

In [ ]:
count_slice['기업규모'].unique()

In [ ]:
# 비중 계산 후 대기업이 좌측에 오도록 정렬
count_share = count_slice.copy()
count_share['비중'] = count_share['활동기업수'] / count_share['활동기업수'].sum()
count_share['기업규모'] = pd.Categorical(count_share['기업규모'], ['대기업', '중소기업'])
count_share = count_share.sort_values('기업규모').reset_index(drop=True)
count_total = int(count_share['활동기업수'].sum())
count_share

### 1-2. 수출 교역액 비중 (2023 · 수출)

In [ ]:
# 2023년 수출 데이터 추출
value_slice = df2[(df2['유형'] == '수출') & (df2['연도'] == 2023)].reset_index(drop=True)

display(value_slice)
value_slice.shape

In [ ]:
value_slice.info()

In [ ]:
# 결측치 확인
value_slice.isna().sum()

In [ ]:
# 중복행 확인
value_slice.duplicated().sum()

In [ ]:
value_slice['기업규모'].unique()

In [ ]:
# 기업규모별 교역액 합계
value_slice.groupby('기업규모')['교역액(천달러)'].sum()

In [ ]:
# 대기업과 그 외(중견+중소+비영리+미연계)로 통합 후 비중 계산
value_raw = value_slice.groupby('기업규모')['교역액(천달러)'].sum()

value_share = pd.DataFrame({
    '구분': ['대기업', '그 외'],
    '교역액(천달러)': [value_raw['대기업'], value_raw.drop('대기업').sum()]
})
value_share['비중'] = value_share['교역액(천달러)'] / value_share['교역액(천달러)'].sum()
value_total = int(value_share['교역액(천달러)'].sum())
value_share

### 1-3. 비중 비교 시각화

In [ ]:
# 색상
COLOR_HL = '#2E4F8F'   # 대기업 (강조)
COLOR_MT = '#D9D9D9'   # 그 외 (연회색)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5))

# 기업수 비중
count_x = count_share['기업규모'].astype(str).tolist()
count_pct = (count_share['비중'] * 100).tolist()
count_n = count_share['활동기업수'].tolist()
count_colors = [COLOR_HL if g == '대기업' else COLOR_MT for g in count_x]

bars = axL.bar(count_x, count_pct, color=count_colors, edgecolor='white', width=0.55)
for bar, pct, n in zip(bars, count_pct, count_n):
    axL.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             f'{pct:.2f}%\n({int(n):,}개)',
             ha='center', va='bottom', fontsize=10)

axL.set_ylim(0, 115)
axL.set_yticks([0, 25, 50, 75, 100])
axL.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
axL.set_ylabel('비중')
axL.set_title('기업수 비중 (2024 · 산업 전체)', fontsize=11, pad=10)
for s in ['top', 'right']:
    axL.spines[s].set_visible(False)

# 수출 교역액 비중
value_x = value_share['구분'].tolist()
value_pct = (value_share['비중'] * 100).tolist()
value_n = value_share['교역액(천달러)'].tolist()
value_colors = [COLOR_HL if g == '대기업' else COLOR_MT for g in value_x]

bars = axR.bar(value_x, value_pct, color=value_colors, edgecolor='white', width=0.55)
for bar, pct, n in zip(bars, value_pct, value_n):
    axR.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 2,
             f'{pct:.1f}%\n({int(n):,} 천달러)',
             ha='center', va='bottom', fontsize=10)

axR.set_ylim(0, 115)
axR.set_yticks([0, 25, 50, 75, 100])
axR.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
axR.set_ylabel('비중')
axR.set_title('수출 교역액 비중 (2023 · 수출)', fontsize=11, pad=10)
for s in ['top', 'right']:
    axR.spines[s].set_visible(False)

# 제목, 출처
fig.suptitle('활동기업의 99.87%가 중소기업, 그러나 수출 교역액의 66%는 대기업',
             fontsize=14, fontweight='bold', y=1.02)
fig.text(0.5, -0.02,
         '출처: 「기업특성별무역통계」, 국가데이터처, 관세청',
         ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

> **거래는 쏠려 있지만, 시도의 흐름은 끊이지 않는다.**

## 2. 시도의 흐름
중소기업 수출은 코로나 시기 일시 위축에도 2015 → 2023년 +15% 성장했다.

### 2-1. 데이터 점검

In [ ]:
# 중소기업 수출 데이터 추출
trend_slice = df2[(df2['유형'] == '수출') & (df2['기업규모'] == '중소기업')].reset_index(drop=True)

display(trend_slice)
trend_slice.shape

In [ ]:
trend_slice.info()

In [ ]:
# 결측치 확인
trend_slice.isna().sum()

In [ ]:
# 중복행 확인
trend_slice.duplicated().sum()

In [ ]:
trend_slice['수출강도'].unique()

### 2-2. 연도별 합계

In [ ]:
# 연도별 합계 (수출강도 4종 합산)
trend_slice.groupby('연도')[['기업수(개)', '교역액(천달러)']].sum()

In [ ]:
# 연도별 시계열 데이터 정리
sme_trend = (
  trend_slice
  .groupby('연도')[['기업수(개)', '교역액(천달러)']]
  .sum()
  .reset_index()
)
sme_trend['기업수(개)'] = sme_trend['기업수(개)'].astype(int)
sme_trend

### 2-3. 시계열 시각화

In [ ]:
# 색상
COLOR_HL = '#2E4F8F'   # 추세선
COLOR_PT = '#D2453A'   # 마지막 연도 강조

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4.5))

last_year = sme_trend['연도'].max()
last_row = sme_trend[sme_trend['연도'] == last_year].iloc[0]

# 수출 참여 중소기업 수 추이
axL.plot(sme_trend['연도'], sme_trend['기업수(개)'],
       marker='o', color=COLOR_HL, linewidth=2, markersize=6)
axL.scatter(last_year, last_row['기업수(개)'], color=COLOR_PT, s=120, zorder=5)
axL.annotate(f'{last_year}년\n{int(last_row["기업수(개)"]):,}개',
           xy=(last_year, last_row['기업수(개)']),
           xytext=(-55, 15), textcoords='offset points',
           fontsize=10, color=COLOR_PT, fontweight='bold')

axL.set_xticks(range(2015, 2024))
axL.set_xlabel('연도')
axL.set_ylabel('기업수(개)')
axL.set_title('수출 참여 중소기업 수', fontsize=11, pad=10)
axL.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axL.grid(axis='y', alpha=0.3)
for s in ['top', 'right']:
  axL.spines[s].set_visible(False)

# 중소기업 수출 교역액 추이
axR.plot(sme_trend['연도'], sme_trend['교역액(천달러)'],
       marker='o', color=COLOR_HL, linewidth=2, markersize=6)
axR.scatter(last_year, last_row['교역액(천달러)'], color=COLOR_PT, s=120, zorder=5)
axR.annotate(f'{last_year}년\n{int(last_row["교역액(천달러)"]):,} 천달러',
           xy=(last_year, last_row['교역액(천달러)']),
           xytext=(-95, 15), textcoords='offset points',
           fontsize=10, color=COLOR_PT, fontweight='bold')

axR.set_xticks(range(2015, 2023))
axR.set_xlabel('연도')
axR.set_ylabel('교역액(천달러)')
axR.set_title('중소기업 수출 교역액', fontsize=11, pad=10)
axR.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axR.grid(axis='y', alpha=0.3)
for s in ['top', 'right']:
  axR.spines[s].set_visible(False)

# 제목, 출처
fig.suptitle('중소기업 수출, 코로나 위축에도 9년간 +15% 성장',
           fontsize=14, fontweight='bold', y=1.02)
fig.text(0.5, -0.02,
       '출처: 「기업특성별무역통계」, 국가데이터처, 관세청',
       ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

---

##  분석 마무리

거래는 여전히 쏠려 있다. 그러나 시도의 흐름은 끊이지 않는다.
**이 격차가 곧 피드 기반 비즈니스 소셜 마켓 플랫폼이 풀려는 문제다.**